# Final Evaluation

Final metrics computation on validation set.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, warnings, math, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torchvision import transforms
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 100

# ── GPU ──────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    print('⚠️  GPU non disponible — l\'inférence sera très lente (~10× plus longue)')
else:
    print(f'✅ GPU détecté : {torch.cuda.get_device_name(0)}')

# ── Chemins ──────────────────────────────────────────────────────
BASE         = '/content/drive/MyDrive/Memoire_Deepfakes'
BENCH_DIR    = f'{BASE}/benchmark_src'
WEIGHTS_DIR  = f'{BASE}/weights/pretrained'
DATA_DIR     = f'{BASE}/data'
SPLITS_DIR   = f'{DATA_DIR}/splits'
RESULTS_DIR  = f'{DATA_DIR}/results'
FIGS_DIR     = f'{RESULTS_DIR}/figures'
os.makedirs(FIGS_DIR, exist_ok=True)

# ── Constantes ───────────────────────────────────────────────────
PROB_COLS    = ['P_Meso4', 'P_XceptionNet', 'P_UCF', 'P_F3Net']
MODEL_NAMES  = ['Meso4',   'XceptionNet',   'UCF',   'F3Net']
LABEL_COL    = 'label'
OUTPUT_COLS  = ['filepath', 'label', 'method', 'split',
                'P_Meso4', 'P_XceptionNet', 'P_UCF', 'P_F3Net']
BATCH_SIZE   = {'Meso4': 32, 'XceptionNet': 32, 'UCF': 2, 'F3Net': 32}
IMG_SIZE     = 256
EXPECTED_VAL = 2811

print(f'\n  BASE    : {BASE}')
print(f'  DEVICE  : {DEVICE}')
print(f'  Val set attendu : {EXPECTED_VAL:,} images')
print()
print('✅ Setup terminé.')


## Load Validation Set

In [ ]:
print('=' * 65)
print('⚠️  OUVERTURE DU COFFRE-FORT — VALIDATION SET')
print('   Date autorisée : 13–16 avril 2026')
print('   Ce chargement constitue un événement unique et irréversible.')
print('=' * 65)

val_manifest_path = f'{SPLITS_DIR}/val_manifest.csv'

if not os.path.isfile(val_manifest_path):
    raise FileNotFoundError(
        f'⚠️  val_manifest.csv introuvable : {val_manifest_path}\n'
        f'    Vérifier le chemin SPLITS_DIR = {SPLITS_DIR}'
    )

val_df = pd.read_csv(val_manifest_path)

print(f'\n  ✅ val_manifest.csv chargé')
print(f'     Lignes      : {len(val_df):,} / {EXPECTED_VAL:,} attendues  '
      f'{"✅" if len(val_df) == EXPECTED_VAL else "⚠️"}')
print(f'     Colonnes    : {list(val_df.columns)}')

dist = val_df[LABEL_COL].value_counts().sort_index().to_dict()
print(f'     Labels      : {dist}  (0=REAL, 1=FAKE)')

if 'method' in val_df.columns:
    print(f'     Méthodes    : {val_df["method"].value_counts().to_dict()}')

# Vérification que val_probs.csv n'existe pas encore
val_probs_path = f'{RESULTS_DIR}/val_probs.csv'
if os.path.isfile(val_probs_path):
    print()
    print('  ⚠️  val_probs.csv EXISTE DÉJÀ — le notebook a déjà été exécuté.')
    print('     Chargement du CSV existant sans relancer l\'inférence.')
    VAL_PROBS_ALREADY_DONE = True
else:
    VAL_PROBS_ALREADY_DONE = False
    print()
    print('  ✅ val_probs.csv absent — inférence à lancer (cellule 9)')

y_val = val_df[LABEL_COL].values

print()
print('=' * 65)


## DeepfakeBench Environment

In [ ]:
# ── sys.path ─────────────────────────────────────────────────────
for p in [BENCH_DIR,
          f'{BENCH_DIR}/training',
          f'{BENCH_DIR}/training/detectors']:
    if p not in sys.path and os.path.isdir(p):
        sys.path.insert(0, p)

print('sys.path (benchmark_src entries) :')
for p in sys.path:
    if 'Memoire' in p:
        print(f'  {p}')

# ── Preprocessing (identique NB04) ───────────────────────────────
TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def load_image(path):
    """Charge et prétraite une image. Retourne None en cas d'erreur."""
    try:
        img = Image.open(path).convert('RGB')
        return TRANSFORM(img)
    except Exception:
        return None

# ── Correctif F3Net (identique NB04) ─────────────────────────────
def fix_f3net_state_dict(state_dict):
    """
    Le checkpoint f3net_best.pth stocke la couche de classification sous
    'backbone.last_linear.1.weight/bias' (nn.Sequential) alors que le
    modèle instancié expose 'backbone.last_linear.weight/bias' (nn.Linear).
    Ce remapping corrige le mismatch de façon permanente.
    """
    return {
        k.replace('backbone.last_linear.1.', 'backbone.last_linear.'): v
        for k, v in state_dict.items()
    }

# ── Métriques ─────────────────────────────────────────────────────
def compute_eer(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    fnr = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fpr - fnr)))
    return float((fpr[idx] + fnr[idx]) / 2.0)

def compute_all_metrics(y_true, y_scores, threshold=0.5):
    y_pred = (np.asarray(y_scores) >= threshold).astype(int)
    return {
        'AUC'      : float(roc_auc_score(y_true, y_scores)),
        'Accuracy' : float(accuracy_score(y_true, y_pred)),
        'F1'       : float(f1_score(y_true, y_pred, zero_division=0)),
        'EER'      : compute_eer(y_true, y_scores),
    }

# ── Mécanisme de reprise intermédiaire (anti-déconnexion Colab) ───
def _save_intermediate(model_name, probs_dict):
    tmp = f'{RESULTS_DIR}/tmp_val_{model_name.lower()}_probs.csv'
    pd.DataFrame(
        list(probs_dict.items()), columns=['filepath', f'P_{model_name}']
    ).to_csv(tmp, index=False)

def _load_intermediate(model_name):
    tmp = f'{RESULTS_DIR}/tmp_val_{model_name.lower()}_probs.csv'
    if os.path.isfile(tmp):
        df = pd.read_csv(tmp)
        return dict(zip(df['filepath'], df[f'P_{model_name}']))
    return {}

print('\n✅ Environnement configuré (preprocessing, correctif F3Net, métriques, reprise).')


## Load Frozen Models

In [ ]:
import importlib, importlib.util, types, yaml, inspect, traceback
import torch.nn as nn

TRAINING_DIR   = f'{BENCH_DIR}/training'
detectors_path = f'{TRAINING_DIR}/detectors'
networks_path  = f'{TRAINING_DIR}/networks'
loss_path      = f'{TRAINING_DIR}/loss'
metrics_path   = f'{BASE}/benchmark_src/metrics'
CONFIG_DIR     = f'{TRAINING_DIR}/config/detector'

# ── BLOC A — Registres universels ────────────────────────────────
class _FakeRegistry:
    def __init__(self, name=''):
        self.name = name
        self._registry = {}
    def register_module(self, name=None, force=False, module=None, **kwargs):
        def decorator(cls):
            key = name if name else cls.__name__
            self._registry[key] = cls
            return cls
        if module is not None:
            return decorator(module)
        return decorator
    def build(self, cfg, *args, **kwargs):
        if isinstance(cfg, dict):
            cls = self._registry.get(cfg.get('type'))
            if cls:
                return cls(**{k: v for k, v in cfg.items() if k != 'type'})
        return None
    def __contains__(self, key): return key in self._registry
    def __getitem__(self, key):  return self._registry[key]
    def get(self, key, default=None): return self._registry.get(key, default)

DETECTOR_REG = _FakeRegistry('detector')
BACKBONE_REG = _FakeRegistry('backbone')
LOSS_REG     = _FakeRegistry('loss')
METRIC_REG   = _FakeRegistry('metric')
LOSSFUNC_REG = _FakeRegistry('lossfunc')

class _StubLoss(nn.Module):
    def __init__(self, *args, **kwargs): super().__init__()
    def forward(self, *args, **kwargs): return torch.tensor(0.0)

for _lk in ['cross_entropy','CrossEntropyLoss','bce','BCELoss','bce_with_logits',
            'BCEWithLogitsLoss','am_softmax','contrastive','contrastive_regularization',
            'focal','l1','l1loss','L1Loss','mse','MSELoss','rec_loss']:
    LOSSFUNC_REG._registry[_lk] = _StubLoss

# ── BLOC B — Fonction make_stub ───────────────────────────────────
def make_stub(full_name, package=None, path=None, **attrs):
    stub = types.ModuleType(full_name)
    stub.__package__ = package or full_name.rsplit('.', 1)[0]
    if path:
        stub.__path__ = [path]
        stub.__file__ = f'{path}/__init__.py'
    for k, v in attrs.items():
        setattr(stub, k, v)
    sys.modules[full_name] = stub
    return stub

# ── BLOC C — Paquets principaux ───────────────────────────────────
detectors_pkg = make_stub('detectors', package='detectors', path=detectors_path, DETECTOR=DETECTOR_REG)
networks_pkg  = make_stub('networks',  package='networks',  path=networks_path,  BACKBONE=BACKBONE_REG)
loss_pkg      = make_stub('loss',      package='loss',      path=loss_path,      LOSS=LOSS_REG, LOSSFUNC=LOSSFUNC_REG)
metrics_pkg   = make_stub('metrics',   package='metrics',   path=metrics_path,   METRIC=METRIC_REG, BACKBONE=BACKBONE_REG)

# ── BLOC D — Stubs sous-modules ───────────────────────────────────
det_utils = make_stub('detectors.utils', package='detectors')
setattr(detectors_pkg, 'utils', det_utils)
det_base = make_stub('detectors.base_detector', package='detectors')
setattr(detectors_pkg, 'base_detector', det_base)

metrics_registry = make_stub('metrics.registry', package='metrics',
    BACKBONE=BACKBONE_REG, DETECTOR=DETECTOR_REG,
    LOSS=LOSS_REG, LOSSFUNC=LOSSFUNC_REG, METRIC=METRIC_REG)
setattr(metrics_pkg, 'registry', metrics_registry)

def _dummy_metrics(*args, **kwargs): return {}
metrics_base = make_stub('metrics.base_metrics_class', package='metrics',
    calculate_metrics_for_train=_dummy_metrics,
    calculate_metrics_for_test=_dummy_metrics,
    BaseMetrics=type('BaseMetrics', (), {
        'calculate_metrics_for_train': staticmethod(_dummy_metrics),
        'calculate_metrics_for_test' : staticmethod(_dummy_metrics),
    }))
setattr(metrics_pkg, 'base_metrics_class', metrics_base)
setattr(metrics_pkg, 'utils', make_stub('metrics.utils', package='metrics'))

for sub in ['resnet34','resnet50','resnet101','resnet152','densenet',
            'inception','vit','swin','efficientnet','hrnet','efficientnetb4']:
    s = make_stub(f'networks.{sub}', package='networks')
    setattr(networks_pkg, sub, s)

for sub in ['abstract_loss_func','am_softmax','bce_loss','cross_entropy_loss',
            'contrastive_regularization','consistency_loss','patch_consistency_loss',
            'region_independent_loss','supercontrast_loss','vgg_loss',
            'capsule_loss','js_loss','id_loss','l1_loss']:
    s = make_stub(f'loss.{sub}', package='loss', LOSSFUNC=LOSSFUNC_REG, LOSS=LOSS_REG)
    setattr(loss_pkg, sub, s)

print('✅ Registres et stubs injectés')

# ── BLOC E — Chargement réel des networks ─────────────────────────
def load_module_from_file(full_name, filepath, package):
    try:
        spec = importlib.util.spec_from_file_location(full_name, filepath)
        mod  = importlib.util.module_from_spec(spec)
        mod.__package__ = package
        sys.modules[full_name] = mod
        spec.loader.exec_module(mod)
        return mod, None
    except Exception as e:
        stub = sys.modules.get(full_name) or make_stub(full_name, package=package)
        sys.modules[full_name] = stub
        return stub, str(e)

loaded_networks, failed_networks = [], []
if os.path.isdir(networks_path):
    for fname in sorted(os.listdir(networks_path)):
        if not fname.endswith('.py') or fname == '__init__.py':
            continue
        mod_name  = fname[:-3]
        full_name = f'networks.{mod_name}'
        mod, err  = load_module_from_file(full_name, os.path.join(networks_path, fname), 'networks')
        setattr(networks_pkg, mod_name, mod)
        if err: failed_networks.append(mod_name)
        else:   loaded_networks.append(mod_name)

print(f'✅ networks/ chargés : {loaded_networks}')

base_det_path = f'{detectors_path}/base_detector.py'
if os.path.isfile(base_det_path):
    mod, err = load_module_from_file('detectors.base_detector', base_det_path, 'detectors')
    setattr(detectors_pkg, 'base_detector', mod)

# ── BLOC E2 — Enregistrement backbones ───────────────────────────
def register_backbone(module_name, class_names, keys):
    mod = sys.modules.get(f'networks.{module_name}')
    if mod is None: return
    for cls_name in class_names:
        cls = getattr(mod, cls_name, None)
        if cls is not None:
            for key in keys:
                BACKBONE_REG._registry[key] = cls
            print(f'✅ BACKBONE{keys} → {cls_name}')
            return

register_backbone('xception', ['Xception','XceptionNet','XceptionModel'], ['xception','Xception'])
register_backbone('mesonet',  ['Meso4','MesoNet','MesoInception4'],       ['meso4','mesonet','MesoNet','Meso4'])

for p in [TRAINING_DIR, f'{BASE}/benchmark_src']:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── BLOC G — Configs des 4 modèles ───────────────────────────────
DEFAULTS = {
    'backbone_name':'xception','backbone_config':None,'pretrained':False,
    'num_classes':2,'compression':'c23','train_batchSize':32,'test_batchSize':32,
    'workers':4,'lr':0.0002,'beta1':0.5,'datapath':'',
    'normalize':{'mean':[0.5,0.5,0.5],'std':[0.5,0.5,0.5]},
    'image_size':256,'with_landmark':False,'with_mask':False,'lnum':0,
    'device':'cuda','logdir':'','manualSeed':42,
    'label_dict':{'FAKE':1,'REAL':0},'clip_size':8,
    'frame_num':{'train':1,'test':1,'val':1},'data_manner':'image',
}
MODEL_SPECIFIC = {
    'Meso4'      : {'backbone_name':'meso4'},
    'XceptionNet': {'backbone_name':'xception'},
    'UCF'        : {'backbone_name':'xception',
                    'backbone_config':{'num_classes':2,'inc':3,'dropout':False},
                    'head_type':'SVM','num_clusters':32,'svm_c':1.0},
    'F3Net'      : {'backbone_name':'xception','FAD_Head_size':768,
                    'LFS_window_size':10,'LFS_M':6,'LFS_stride':16},
}
MODEL_CONFIGS = [
    {'name':'Meso4',       'filepath':f'{detectors_path}/meso4_detector.py',
     'class_name':'Meso4Detector',   'weight_file':f'{WEIGHTS_DIR}/meso4_best.pth',
     'config_file':f'{CONFIG_DIR}/meso4.yaml'},
    {'name':'XceptionNet', 'filepath':f'{detectors_path}/xception_detector.py',
     'class_name':'XceptionDetector','weight_file':f'{WEIGHTS_DIR}/xception_best.pth',
     'config_file':f'{CONFIG_DIR}/xception.yaml'},
    {'name':'UCF',         'filepath':f'{detectors_path}/ucf_detector.py',
     'class_name':'UCFDetector',     'weight_file':f'{WEIGHTS_DIR}/ucf_best.pth',
     'config_file':f'{CONFIG_DIR}/ucf.yaml'},
    {'name':'F3Net',       'filepath':f'{detectors_path}/f3net_detector.py',
     'class_name':'F3netDetector',   'weight_file':f'{WEIGHTS_DIR}/f3net_best.pth',
     'config_file':f'{CONFIG_DIR}/f3net.yaml'},
]

# ── BLOC H — Boucle de chargement ────────────────────────────────
_original_torch_load = torch.load

class _DummyStateDict:
    def __getitem__(self, key):       return torch.zeros(64,3,3,3)
    def __contains__(self, key):      return True
    def get(self, key, default=None): return torch.zeros(64,3,3,3)
    def items(self):  return {}.items()
    def keys(self):   return {}.keys()
    def values(self): return {}.values()

def _safe_torch_load(f, *args, **kwargs):
    if isinstance(f, (bool, type(None))): return _DummyStateDict()
    if isinstance(f, str) and not os.path.isfile(f): return _DummyStateDict()
    return _original_torch_load(f, *args, **kwargs)

torch.load = _safe_torch_load
loaded_models = {}

print()
print('=' * 65)
print('CHARGEMENT DES 4 MODÈLES')
print('=' * 65)

for cfg in MODEL_CONFIGS:
    name = cfg['name']
    print(f'\n  [{name}]')
    try:
        model_config = yaml.safe_load(open(cfg['config_file'])) or {}
    except Exception:
        model_config = {}

    module_name = f"detectors.{os.path.basename(cfg['filepath'])[:-3]}"
    mod, err = load_module_from_file(module_name, cfg['filepath'], 'detectors')
    setattr(detectors_pkg, os.path.basename(cfg['filepath'])[:-3], mod)
    if err:
        print(f'    ⚠️  Import échoué : {err}'); continue

    ModelClass = getattr(mod, cfg['class_name'], None)
    if ModelClass is None:
        print(f'    ⚠️  Classe {cfg["class_name"]} introuvable'); continue

    if name == 'F3Net':
        def _safe_f3net_build_backbone(self, config):
            backbone_class = BACKBONE_REG._registry.get('xception')
            return backbone_class({'num_classes':config.get('num_classes',2),
                                   'mode':'original','inc':12,'dropout':False})
        ModelClass.build_backbone = _safe_f3net_build_backbone

    merged_config = {**DEFAULTS, **MODEL_SPECIFIC.get(name, {}), **model_config}
    merged_config['pretrained'] = False

    try:
        model = ModelClass(merged_config)
    except Exception as e:
        print(f'    ⚠️  Instanciation échouée : {e}'); traceback.print_exc(); continue

    ckpt = _original_torch_load(cfg['weight_file'], map_location='cpu')
    if isinstance(ckpt, dict):
        for key in ['state_dict','model','net','params']:
            if key in ckpt:
                state_dict = ckpt[key]; break
        else:
            state_dict = ckpt
    else:
        state_dict = ckpt

    state_dict = {(k[len('module.'):] if k.startswith('module.') else k): v
                  for k, v in state_dict.items()}

    # Correctif F3Net
    if name == 'F3Net':
        state_dict = fix_f3net_state_dict(state_dict)
        print('    Correctif F3Net last_linear.1 → last_linear appliqué')

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f'    Missing={len(missing)}  Unexpected={len(unexpected)}')

    model.eval()
    model.to(DEVICE)
    loaded_models[name] = model
    print(f'    ✅ Chargé sur {DEVICE}')

torch.load = _original_torch_load
print()
print(f'✅ {len(loaded_models)}/4 modèles chargés : {list(loaded_models.keys())}')

## Inference on Validation Set

In [ ]:
import torchvision.transforms as T
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

if VAL_PROBS_ALREADY_DONE:
    print('⚠️  val_probs.csv déjà présent — inférence ignorée.')
else:
    # ── Transform ────────────────────────────────────────────────
    TRANSFORM = T.Compose([
        T.Resize((256, 256),
                 interpolation=T.InterpolationMode.BILINEAR,
                 antialias=True),
        T.ToTensor(),
        T.Normalize(mean=[0.5, 0.5, 0.5],
                    std =[0.5, 0.5, 0.5]),
    ])

    # ── Dataset ──────────────────────────────────────────────────
    class FaceDataset(Dataset):
        def __init__(self, df, transform):
            self.records   = df[['filepath']].reset_index(drop=True)
            self.transform = transform
            self.n_errors  = 0
        def __len__(self):
            return len(self.records)
        def __getitem__(self, idx):
            filepath = str(self.records.iloc[idx]['filepath'])
            try:
                img    = Image.open(filepath).convert('RGB')
                tensor = self.transform(img)
            except Exception:
                self.n_errors += 1
                tensor = torch.zeros(3, 256, 256)
            return tensor, filepath

    def make_dataloader(df, batch_size):
        dataset = FaceDataset(df, TRANSFORM)
        return DataLoader(
            dataset,
            batch_size  = batch_size,
            shuffle     = False,
            num_workers = 0,
            pin_memory  = torch.cuda.is_available(),
            drop_last   = False,
        )

    # ── Extraction P(FAKE) ────────────────────────────────────────
    def _extract_prob_fake(output):
        if isinstance(output, dict):
            logits = output.get('cls',
                     output.get('logits',
                     output.get('pred',
                     list(output.values())[0])))
        else:
            logits = output
        if isinstance(logits, (tuple, list)):
            logits = logits[0]
        if logits.ndim == 1:
            logits = logits.unsqueeze(0)
        n_classes = logits.shape[-1]
        if n_classes == 2:
            return F.softmax(logits, dim=1)[:, 1]
        elif n_classes == 1:
            return torch.sigmoid(logits).squeeze(1)
        else:
            return torch.sigmoid(logits[:, -1])

    # ── Forward pass ──────────────────────────────────────────────
    def _forward_one_batch(model, imgs, device):
        n_batch = imgs.shape[0]
        with torch.no_grad():
            try:
                data_dict = {
                    'image'   : imgs,
                    'label'   : torch.zeros(n_batch, dtype=torch.long).to(device),
                    'mask'    : None,
                    'landmark': None,
                }
                output = model(data_dict)
            except (TypeError, KeyError, AttributeError):
                output = model(imgs)
        return _extract_prob_fake(output)

    # ── Sauvegarde / reprise intermédiaire ───────────────────────
    def _save_intermediate(probs_dict, model_name):
        col  = f'P_{model_name}'
        df   = pd.DataFrame([{'filepath': k, col: v}
                              for k, v in probs_dict.items()])
        path = f'{RESULTS_DIR}/tmp_val_{model_name.lower()}_probs.csv'
        df.to_csv(path, index=False)

    def _load_intermediate(model_name):
        col  = f'P_{model_name}'
        path = f'{RESULTS_DIR}/tmp_val_{model_name.lower()}_probs.csv'
        if os.path.isfile(path):
            df = pd.read_csv(path)
            if 'filepath' in df.columns and col in df.columns:
                return dict(zip(df['filepath'].tolist(), df[col].tolist()))
        return {}

    # ── Boucle principale ─────────────────────────────────────────
    BATCH_SIZE_DEFAULT = 32
    BATCH_SIZE_UCF     = 2
    all_probs          = {}

    for model_name in MODEL_NAMES:
        print(f'\n  ━━━━ {model_name} ━━━━')
        t_start = time.time()

        # Reprise depuis intermédiaire si disponible
        probs_dict = _load_intermediate(model_name)
        if probs_dict:
            print(f'     ♻️  Reprise : {len(probs_dict):,} prédictions chargées')
            all_probs[model_name] = probs_dict
            continue

        model      = loaded_models[model_name]
        batch_size = BATCH_SIZE_UCF if model_name == 'UCF' else BATCH_SIZE_DEFAULT
        loader     = make_dataloader(val_df, batch_size)
        probs_dict = {}
        error_list = []

        for imgs, filepaths in tqdm(loader,
                                    desc=f'  {model_name:<12s} [val]',
                                    unit='batch'):
            actual_bs      = imgs.shape[0]
            needs_truncate = False

            # Padding UCF si batch de 1 image
            if model_name == 'UCF' and actual_bs < 2:
                imgs           = torch.cat([imgs, imgs], dim=0)
                needs_truncate = True

            imgs = imgs.to(DEVICE)

            try:
                probs_tensor = _forward_one_batch(model, imgs, DEVICE)
                probs_list   = probs_tensor.cpu().tolist()

                if needs_truncate:
                    probs_list = probs_list[:actual_bs]
                    filepaths  = list(filepaths)[:actual_bs]

                for fp, prob in zip(filepaths, probs_list):
                    probs_dict[str(fp)] = float(prob)

            except Exception as exc:
                for fp in list(filepaths)[:actual_bs]:
                    probs_dict[str(fp)] = float('nan')
                    error_list.append((str(fp), str(exc)))

        elapsed    = time.time() - t_start
        probs_arr  = np.array(list(probs_dict.values()))
        n_nan      = int(np.isnan(probs_arr).sum())

        print(f'     Images    : {len(probs_dict):,} / {len(val_df):,}')
        print(f'     NaN       : {n_nan}')
        print(f'     Durée     : {elapsed/60:.1f} min')
        print(f'     P(FAKE)   : min={np.nanmin(probs_arr):.4f}  '
              f'max={np.nanmax(probs_arr):.4f}  '
              f'mean={np.nanmean(probs_arr):.4f}')
        if error_list:
            print(f'     ⚠️  {len(error_list)} erreur(s) forward pass')

        _save_intermediate(probs_dict, model_name)
        print(f'     ✅ Intermédiaire sauvegardé')
        all_probs[model_name] = probs_dict

    print()
    print('✅ Inférence terminée pour les 4 modèles.')

In [ ]:
# ================================================================
# CELLULE 12 — Assemblage et sauvegarde de val_probs.csv
# ================================================================

if VAL_PROBS_ALREADY_DONE:
    val_probs_df = pd.read_csv(val_probs_path)
    print(f'✅ val_probs.csv rechargé ({len(val_probs_df):,} lignes)')
else:
    df_out = val_df.copy()

    _probs_registry = {
        'Meso4'      : 'P_Meso4',
        'XceptionNet': 'P_XceptionNet',
        'UCF'        : 'P_UCF',
        'F3Net'      : 'P_F3Net',
    }

    for model_name, col in _probs_registry.items():
        probs_d = all_probs.get(model_name, {})
        if not probs_d:
            # Dernière tentative de reprise
            probs_d = _load_intermediate(model_name)
        df_out[col] = df_out['filepath'].map(probs_d)

    # Sélection colonnes
    available = [c for c in OUTPUT_COLS if c in df_out.columns]
    df_out    = df_out[available].copy()

    # Diagnostic NaN
    prob_cols = [c for c in available if c.startswith('P_')]
    total_nan = 0
    print('Vérification NaN :')
    for col in prob_cols:
        n_nan = int(df_out[col].isna().sum())
        total_nan += n_nan
        flag = '✅' if n_nan == 0 else f'⚠️  {n_nan} NaN'
        print(f'  {flag}  {col}')

    df_out.to_csv(val_probs_path, index=False)
    val_probs_df = df_out

    print(f'\n✅ val_probs.csv sauvegardé : {val_probs_path}')
    print(f'   Lignes   : {len(df_out):,}')
    print(f'   Colonnes : {list(df_out.columns)}')
    print(f'   NaN total : {total_nan}')

    # Nettoyage intermédiaires
    print('\nNettoyage fichiers intermédiaires :')
    for mn in MODEL_NAMES:
        tmp = f'{RESULTS_DIR}/tmp_val_{mn.lower()}_probs.csv'
        if os.path.isfile(tmp):
            os.remove(tmp)
            print(f'  🗑  {os.path.basename(tmp)}')

# Vecteur labels val
y_val = val_probs_df[LABEL_COL].values
print(f'\ny_val shape : {y_val.shape} | Distribution : {dict(zip(*np.unique(y_val, return_counts=True)))}')


## Recalibrate Ensemble Parameters

In [ ]:
train_probs_path = f'{RESULTS_DIR}/train_probs.csv'
if not os.path.isfile(train_probs_path):
    raise FileNotFoundError(f'train_probs.csv introuvable : {train_probs_path}')

train_df = pd.read_csv(train_probs_path)
X_train  = train_df[PROB_COLS].values
y_train  = train_df[LABEL_COL].values

print('=' * 65)
print('RECALIBRATION DES PARAMÈTRES ENSEMBLE')
print('=' * 65)

# ── Scénario B — poids ───────────────────────────────────────────
print('\n  Scénario B — Poids (accuracy Train Set) :')
raw_w = {}
for col in PROB_COLS:
    pred = (train_df[col].values >= 0.5).astype(int)
    acc  = float(accuracy_score(y_train, pred))
    raw_w[col] = acc
    print(f'    {col:<18s} : {acc:.4f}')

total_w   = sum(raw_w.values())
norm_w_B  = {col: w / total_w for col, w in raw_w.items()}
print('  Poids normalisés :', {k: round(v, 4) for k, v in norm_w_B.items()})

# ── Scénario C — Méta-learner ─────────────────────────────────────
print('\n  Scénario C — Régression logistique (Train Set) :')
clf = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
clf.fit(X_train, y_train)

nb05_coefs = {
    'Meso4': -1.9887, 'XceptionNet': 1.7925,
    'UCF'  : 1.1870,  'F3Net'      : 1.1773
}
print('  Coefficients (NB06 vs NB05) :')
for model, col, coef in zip(MODEL_NAMES, PROB_COLS, clf.coef_[0]):
    ref  = nb05_coefs.get(model, 0)
    diff = abs(coef - ref)
    flag = '✅' if diff < 0.001 else '⚠️'
    print(f'  {flag} {model:<14s} β={coef:+.4f}  (NB05: {ref:+.4f}  Δ={diff:.4f})')
print(f'  Intercept : β0 = {clf.intercept_[0]:+.4f}')

print()
print('✅ Paramètres Scén. B et C recalibrés.')


## Complete Evaluation

In [ ]:
X_val = val_probs_df[PROB_COLS].values
RESULTS_VAL = {}

print('=' * 70)
print('ÉVALUATION FINALE — VALIDATION SET (2 811 images inédites)')
print('=' * 70)

# ── Modèles individuels ───────────────────────────────────────────
print('\n── Modèles individuels ──')
for model, col in zip(MODEL_NAMES, PROB_COLS):
    s = val_probs_df[col].values
    m = compute_all_metrics(y_val, s)
    RESULTS_VAL[model] = {'val': m, 'scores': s}
    print(f'  {model:<14s} AUC={m["AUC"]:.4f}  Acc={m["Accuracy"]:.4f}  '
          f'F1={m["F1"]:.4f}  EER={m["EER"]:.4f}')

# ── Scénario A — Vote majoritaire ────────────────────────────────
print('\n── Scénario A — Vote majoritaire ──')
votes_A    = (val_probs_df[PROB_COLS].values >= 0.5).astype(int)
n_votes_A  = votes_A.sum(axis=1)
score_A    = val_probs_df[PROB_COLS].mean(axis=1).values
pred_A     = (n_votes_A >= 2).astype(int)
m_A        = compute_all_metrics(y_val, score_A)
m_A['Accuracy'] = float(accuracy_score(y_val, pred_A))
m_A['F1']       = float(f1_score(y_val, pred_A, zero_division=0))
RESULTS_VAL['Scenario_A'] = {'val': m_A, 'scores': score_A, 'preds': pred_A}
dist_A = {v: int((n_votes_A == v).sum()) for v in range(5)}
print(f'  AUC={m_A["AUC"]:.4f}  Acc={m_A["Accuracy"]:.4f}  '
      f'F1={m_A["F1"]:.4f}  EER={m_A["EER"]:.4f}')
print(f'  Distribution votes FAKE : {dist_A}')

# ── Scénario B — Moyenne pondérée ────────────────────────────────
print('\n── Scénario B — Moyenne pondérée ──')
score_B = sum(norm_w_B[col] * val_probs_df[col].values for col in PROB_COLS)
m_B     = compute_all_metrics(y_val, score_B)
RESULTS_VAL['Scenario_B'] = {'val': m_B, 'scores': score_B}
print(f'  AUC={m_B["AUC"]:.4f}  Acc={m_B["Accuracy"]:.4f}  '
      f'F1={m_B["F1"]:.4f}  EER={m_B["EER"]:.4f}')

# ── Scénario C — Méta-learner ─────────────────────────────────────
print('\n── Scénario C — Méta-Learner ──')
score_C = clf.predict_proba(X_val)[:, 1]
m_C     = compute_all_metrics(y_val, score_C)
RESULTS_VAL['Scenario_C'] = {'val': m_C, 'scores': score_C}
print(f'  AUC={m_C["AUC"]:.4f}  Acc={m_C["Accuracy"]:.4f}  '
      f'F1={m_C["F1"]:.4f}  EER={m_C["EER"]:.4f}')

print()
print('=' * 70)
print('✅ Évaluation finale complète.')


## Ensemble Probabilities

In [ ]:
import pandas as pd

print('Scores du Scénario A (Vote majoritaire):')
display(pd.Series(RESULTS_VAL['Scenario_A']['scores']).describe())

print('\nScores du Scénario B (Moyenne pondérée):')
display(pd.Series(RESULTS_VAL['Scenario_B']['scores']).describe())

print('\nScores du Scénario C (Méta-Learner):')
display(pd.Series(RESULTS_VAL['Scenario_C']['scores']).describe())

In [ ]:
import pandas as pd

ensemble_probs_df = pd.DataFrame({
    'y_true': y_val,
    'P_Scenario_A': RESULTS_VAL['Scenario_A']['scores'],
    'P_Scenario_B': RESULTS_VAL['Scenario_B']['scores'],
    'P_Scenario_C': RESULTS_VAL['Scenario_C']['scores']
})

ensemble_probs_filepath = f'{RESULTS_DIR}/ensemble_scenario_probs_val.csv'
ensemble_probs_df.to_csv(ensemble_probs_filepath, index=False)

print(f'✅ Probabilités des scénarios ensemble sauvegardées ici : {ensemble_probs_filepath}')
display(ensemble_probs_df.head())

## Test vs Validation Comparison

In [ ]:
# Chargement des métriques Test depuis NB05
summary_path = f'{RESULTS_DIR}/metrics_summary.csv'
test_summary = pd.read_csv(summary_path)
test_rows    = test_summary[test_summary['Split'] == 'Test'].copy()

DISPLAY_NAMES = {
    'Meso4'      : 'Meso4',
    'XceptionNet': 'XceptionNet',
    'UCF'        : 'UCF',
    'F3Net'      : 'F3Net',
    'Scenario_A' : 'Scén. A — Vote maj.',
    'Scenario_B' : 'Scén. B — Moy. pond.',
    'Scenario_C' : 'Scén. C — Méta-Learn.',
}

DISPLAY_TO_KEY = {
    'Meso4'               : 'Meso4',
    'XceptionNet'         : 'XceptionNet',
    'UCF'                 : 'UCF',
    'F3Net'               : 'F3Net',
    'Scén. A — Vote maj.' : 'Scenario_A',
    'Scén. B — Moy. pond.': 'Scenario_B',
    'Scén. C — Méta-Learn.': 'Scenario_C',
}

rows = []
METRICS = ['AUC', 'Accuracy', 'F1', 'EER']

for _, test_row in test_rows.iterrows():
    disp = test_row['Modèle / Scénario']
    key  = DISPLAY_TO_KEY.get(disp, disp)

    if key not in RESULTS_VAL:
        continue

    val_m = RESULTS_VAL[key]['val']
    row   = {'Modèle / Scénario': disp, 'Catégorie': test_row['Catégorie']}

    for metric in METRICS:
        t_val = test_row[metric]
        v_val = val_m[metric]
        delta = v_val - t_val
        row[f'{metric}_Test'] = round(t_val, 4)
        row[f'{metric}_Val']  = round(v_val, 4)
        row[f'{metric}_Δ']    = round(delta, 4)

    rows.append(row)

final_df = pd.DataFrame(rows)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:+.4f}'.format)

print('=' * 120)
print('TABLEAU FINAL — Métriques Test vs Validation (Δ = Val − Test)')
print('Signe Δ : positif = meilleure généralisation | négatif = dégradation')
print('=' * 120)
print(final_df.to_string(index=False))
print('=' * 120)

# Sauvegarde
final_path = f'{RESULTS_DIR}/final_metrics_val.csv'
final_df.to_csv(final_path, index=False)
print(f'\n✅ Tableau final sauvegardé : {final_path}')


## Summary Statistics

In [ ]:
import pandas as pd
import numpy as np

summary_rows = []

# Regrouper les informations de RESULTS_VAL et final_df
for key, data in RESULTS_VAL.items():
    row = {'Modèle / Scénario': key}

    # Ajouter les métriques d'évaluation sur le Validation Set
    for metric, value in data['val'].items():
        row[f'{metric}_Val'] = value

    # Ajouter les statistiques des scores de prédiction sur le Validation Set
    scores = data['scores']
    row['Score_Mean_Val'] = np.nanmean(scores)
    row['Score_Min_Val']  = np.nanmin(scores)
    row['Score_Max_Val']  = np.nanmax(scores)

    summary_rows.append(row)

# Créer le DataFrame final
full_summary_df = pd.DataFrame(summary_rows)

# Assurer un ordre des colonnes plus lisible
ordered_columns = [
    'Modèle / Scénario',
    'AUC_Val', 'Accuracy_Val', 'F1_Val', 'EER_Val',
    'Score_Mean_Val', 'Score_Min_Val', 'Score_Max_Val'
]
full_summary_df = full_summary_df[ordered_columns]

# Renommer les noms affichables pour les scénarios
display_name_map = {
    'Meso4': 'Meso4',
    'XceptionNet': 'XceptionNet',
    'UCF': 'UCF',
    'F3Net': 'F3Net',
    'Scenario_A': 'Scén. A — Vote maj.',
    'Scenario_B': 'Scén. B — Moy. pond.',
    'Scenario_C': 'Scén. C — Méta-Learn.'
}
full_summary_df['Modèle / Scénario'] = full_summary_df['Modèle / Scénario'].map(display_name_map)


pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

print('=' * 120)
print('TABLEAU RÉCAPITULATIF COMPLET — Métriques et Statistiques des Scores (Validation Set)')
print('=' * 120)
print(full_summary_df.to_string(index=False))
print('=' * 120)

# Sauvegarde du tableau complet
full_summary_path = f'{RESULTS_DIR}/full_validation_summary.csv'
full_summary_df.to_csv(full_summary_path, index=False)
print(f'\n✅ Tableau récapitulatif complet sauvegardé : {full_summary_path}')

## Visualizations

In [ ]:
COLORS_INDIV = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
COLORS_ENS   = ['#9467bd', '#8c564b', '#e377c2']

# ── Figure 1 — Courbes ROC (Val Set) ─────────────────────────────
fig1, ax = plt.subplots(figsize=(8, 7))

for model, color in zip(MODEL_NAMES, COLORS_INDIV):
    fpr, tpr, _ = roc_curve(y_val, RESULTS_VAL[model]['scores'])
    auc_v = RESULTS_VAL[model]['val']['AUC']
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{model} (AUC = {auc_v:.3f})')

ens_config = [
    ('Scén. A — Vote maj.',   'Scenario_A', COLORS_ENS[0], '--'),
    ('Scén. B — Moy. pond.',  'Scenario_B', COLORS_ENS[1], '-.'),
    ('Scén. C — Méta-Learn.', 'Scenario_C', COLORS_ENS[2], ':'),
]
for label_e, key, color, ls in ens_config:
    fpr, tpr, _ = roc_curve(y_val, RESULTS_VAL[key]['scores'])
    auc_v = RESULTS_VAL[key]['val']['AUC']
    ax.plot(fpr, tpr, color=color, lw=2.5, ls=ls,
            label=f'{label_e} (AUC = {auc_v:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Aléatoire (AUC = 0.500)')
ax.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
ax.set_ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
ax.set_title(
    'Courbes ROC — Validation Set (RÉSULTATS DÉFINITIFS)\n'
    'Modèles individuels [poids gelés FF++ c23] vs Scénarios Ensemble',
    fontsize=12
)
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()

roc_path = f'{FIGS_DIR}/roc_curves_val_FINAL.png'
fig1.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure ROC finale sauvegardée : {roc_path}')

# ── Figure 2 — Matrices de Confusion (Val Set) ───────────────────
cm_items = [
    ('Meso4',              (val_probs_df['P_Meso4'].values       >= 0.5).astype(int)),
    ('XceptionNet',        (val_probs_df['P_XceptionNet'].values >= 0.5).astype(int)),
    ('UCF',                (val_probs_df['P_UCF'].values         >= 0.5).astype(int)),
    ('F3Net',              (val_probs_df['P_F3Net'].values        >= 0.5).astype(int)),
    ('Scén. A — Vote maj.',   RESULTS_VAL['Scenario_A']['preds']),
    ('Scén. B — Moy. pond.',  (RESULTS_VAL['Scenario_B']['scores'] >= 0.5).astype(int)),
    ('Scén. C — Méta-Learn.', (RESULTS_VAL['Scenario_C']['scores'] >= 0.5).astype(int)),
]

fig2, axes = plt.subplots(2, 4, figsize=(18, 9))
axes_flat  = axes.flatten()

for idx, (title, preds) in enumerate(cm_items):
    cm   = confusion_matrix(y_val, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['REAL', 'FAKE'])
    disp.plot(ax=axes_flat[idx], colorbar=False, cmap='Blues')
    acc_v = accuracy_score(y_val, preds)
    f1_v  = f1_score(y_val, preds, zero_division=0)
    axes_flat[idx].set_title(f'{title}\nAcc={acc_v:.3f}  F1={f1_v:.3f}', fontsize=9)

axes_flat[7].axis('off')
fig2.suptitle(
    'Matrices de Confusion — Validation Set — RÉSULTATS DÉFINITIFS (seuil = 0.5)\n'
    'Ligne 1 : Modèles individuels  │  Ligne 2 : Scénarios Ensemble',
    fontsize=12, y=1.01
)
plt.tight_layout()

cm_path = f'{FIGS_DIR}/confusion_matrices_val_FINAL.png'
fig2.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure CM finale sauvegardée : {cm_path}')


## Final Summary

In [ ]:
print('=' * 70)
print('RÉSUMÉ FINAL — NOTEBOOK 06 — ÉVALUATION DÉFINITIVE')
print('=' * 70)

print('\n  RÉSULTATS VAL SET — Métriques finales :')
print(f'  {"Modèle/Scénario":<25s} {"AUC":>8s} {"Acc":>8s} {"F1":>8s} {"EER":>8s}')
print('  ' + '-' * 60)

ENTRY_ORDER = [
    ('Meso4',      'Meso4'),
    ('XceptionNet','XceptionNet'),
    ('UCF',        'UCF'),
    ('F3Net',      'F3Net'),
    ('Scén. A',    'Scenario_A'),
    ('Scén. B',    'Scenario_B'),
    ('Scén. C',    'Scenario_C'),
]

for label, key in ENTRY_ORDER:
    m = RESULTS_VAL[key]['val']
    print(f'  {label:<25s} '
          f'{m["AUC"]:>8.4f} '
          f'{m["Accuracy"]:>8.4f} '
          f'{m["F1"]:>8.4f} '
          f'{m["EER"]:>8.4f}')

# ── Vérifications intégrité ───────────────────────────────────────
print()
print('  Vérifications intégrité :')

checks = {
    'val_probs.csv généré'        : os.path.isfile(val_probs_path),
    'final_metrics_val.csv généré': os.path.isfile(final_path),
    'ROC val sauvegardée'         : os.path.isfile(roc_path),
    'CM val sauvegardée'          : os.path.isfile(cm_path),
    'Aucun tmp_val_* résiduel'    : not any(
        os.path.isfile(f'{RESULTS_DIR}/tmp_val_{mn.lower()}_probs.csv')
        for mn in MODEL_NAMES
    ),
    'Lignes val correctes (2811)' : len(pd.read_csv(val_probs_path)) == EXPECTED_VAL,
}

all_ok = True
for desc, ok in checks.items():
    flag = '✅' if ok else '⚠️'
    print(f'    {flag}  {desc}')
    if not ok:
        all_ok = False

print()
print('=' * 70)
if all_ok:
    print('  ✅ NOTEBOOK 06 TERMINÉ AVEC SUCCÈS')
    print()
    print('  Fichiers générés dans data/results/ :')
    print('    → val_probs.csv              (2 811 lignes × 8 colonnes)')
    print('    → final_metrics_val.csv      (tableau Test vs Val)')
    print('    → figures/roc_curves_val_FINAL.png')
    print('    → figures/confusion_matrices_val_FINAL.png')
    print()
    print('  Phase 3 — COMPLÈTE. Prochaine étape : Phase 4 — Rédaction.')
else:
    print('  ⚠️  PROBLÈMES DÉTECTÉS — Vérifier les points marqués ⚠️  ci-dessus')
print('=' * 70)
